# Module BM25 — Solution

Retrieve the most relevant of 3 PDFs for two queries, using **two methods** and comparing them:

1. **BM25** — lexical ranking on tokenized text (`rank_bm25`).
2. **MiniLM** — semantic ranking with sentence embeddings (`all-MiniLM-L6-v2`).

Both run on CPU and need no Hugging Face token.

In [18]:
!pip install -q rank_bm25 sentence-transformers pypdf

In [19]:
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
import numpy as np
import os
import re

## 1. Extract text from each PDF

In [20]:
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + " "
    return text

## 2. Load the 3 PDFs and tokenize (lowercase split)

In [21]:
pdf_dir = "/kaggle/input/datasets/optimalbhandari/pdf-dataset"

pdf_files = [
    os.path.join(pdf_dir, "learning-langchain-for-true-epub-9781098167288.pdf"),
    os.path.join(pdf_dir, "Natural-Language-Processing-Python.pdf"),
    os.path.join(pdf_dir, "test_doc.pdf"),
]
names = [os.path.basename(p) for p in pdf_files]  # short names for printing


def tokenize(text):
    """Lowercase and keep only word characters. Plain .split() leaves punctuation
    attached (e.g. 'tokenization?'), which then matches nothing in the PDFs."""
    return re.findall(r"[a-z0-9]+", text.lower())


# Treat each PDF as one document
documents = [extract_text_from_pdf(pdf) for pdf in pdf_files]

# Tokenize the documents (punctuation stripped)
tokenized_documents = [tokenize(doc) for doc in documents]

print(f"Loaded {len(documents)} PDFs.")
for n, toks in zip(names, tokenized_documents):
    print(f"  {n}: {len(toks)} tokens")

Loaded 3 PDFs.
  learning-langchain-for-true-epub-9781098167288.pdf: 137 tokens
  Natural-Language-Processing-Python.pdf: 183 tokens
  test_doc.pdf: 68 tokens


## 3. The two queries (tokenized)

In [22]:
queries = [
    "What is Tokenization?",
    "What are different Types of Tokenization?",
]

for i, q in enumerate(queries, 1):
    print(f"Query {i}: {q}")
    print("  naive split:", q.lower().split())   # leaves the '?' attached
    print("  cleaned    :", tokenize(q))          # used for BM25

Query 1: What is Tokenization?
  naive split: ['what', 'is', 'tokenization?']
  cleaned    : ['what', 'is', 'tokenization']
Query 2: What are different Types of Tokenization?
  naive split: ['what', 'are', 'different', 'types', 'of', 'tokenization?']
  cleaned    : ['what', 'are', 'different', 'types', 'of', 'tokenization']


## 4. BM25 retrieval

BM25 ranks each PDF by lexical overlap with the tokenized query.

In [23]:
bm25 = BM25Okapi(tokenized_documents)

for q in queries:
    scores = bm25.get_scores(tokenize(q))
    ranking = sorted(zip(names, scores), key=lambda x: x[1], reverse=True)
    print(f"Query: {q}")
    for name, s in ranking:
        print(f"  {s:.4f}  {name}")
    print()

Query: What is Tokenization?
  1.1695  Natural-Language-Processing-Python.pdf
  0.1217  learning-langchain-for-true-epub-9781098167288.pdf
  0.1142  test_doc.pdf

Query: What are different Types of Tokenization?
  3.0102  Natural-Language-Processing-Python.pdf
  0.0861  test_doc.pdf
  0.0660  learning-langchain-for-true-epub-9781098167288.pdf



## 5. MiniLM semantic retrieval

`all-MiniLM-L6-v2` embeds each PDF and each query, then ranks by cosine similarity —
so it can match meaning even when the exact words differ.

In [24]:
model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = model.encode(documents, convert_to_tensor=True)

for q in queries:
    q_emb = model.encode(q, convert_to_tensor=True)
    sims = util.cos_sim(q_emb, doc_embeddings)[0]
    ranking = sorted(zip(names, sims.tolist()), key=lambda x: x[1], reverse=True)
    print(f"Query: {q}")
    for name, s in ranking:
        print(f"  {s:.4f}  {name}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: What is Tokenization?
  0.5613  Natural-Language-Processing-Python.pdf
  0.3837  learning-langchain-for-true-epub-9781098167288.pdf
  0.1292  test_doc.pdf

Query: What are different Types of Tokenization?
  0.6010  Natural-Language-Processing-Python.pdf
  0.3751  learning-langchain-for-true-epub-9781098167288.pdf
  0.0377  test_doc.pdf



## 6. Show the actual text (passage-level retrieval)

Document ranking only tells us *which* PDF wins. To see *why*, split each PDF into
sentences and retrieve the best-matching sentences with MiniLM,so we read the real text,
not just a filename.

In [25]:
# Split every PDF into sentences, remembering which file each came from.
passages, passage_src = [], []
for name, doc in zip(names, documents):
    for sent in re.split(r"(?<=[.!?])\s+", doc):
        sent = sent.strip()
        if len(sent.split()) >= 4:        # skip tiny fragments
            passages.append(sent)
            passage_src.append(name)

passage_emb = model.encode(passages, convert_to_tensor=True)

for q in queries:
    sims = util.cos_sim(model.encode(q, convert_to_tensor=True), passage_emb)[0]
    top = sims.topk(2)
    print(f"Query: {q}")
    for score, i in zip(top.values, top.indices):
        i = int(i)
        print(f"  [{score:.3f}] ({passage_src[i]}) {passages[i]}")
    print()

Query: What is Tokenization?
  [0.836] (Natural-Language-Processing-Python.pdf) There are several different types of tokenization.
  [0.818] (Natural-Language-Processing-Python.pdf) Tokenization is the
process of breaking a piece of text into smaller units called tokens.

Query: What are different Types of Tokenization?
  [0.943] (Natural-Language-Processing-Python.pdf) There are several different types of tokenization.
  [0.721] (Natural-Language-Processing-Python.pdf) A token can be a word, a number, a
punctuation mark, or a subword fragment.



## 7. Compare which PDF each method picks

In [26]:
for q in queries:
    bm25_top = names[int(np.argmax(bm25.get_scores(tokenize(q))))]
    sims = util.cos_sim(model.encode(q, convert_to_tensor=True), doc_embeddings)[0]
    mini_top = names[int(sims.argmax())]
    print(f"Query: {q}")
    print(f"  BM25   top: {bm25_top}")
    print(f"  MiniLM top: {mini_top}")
    print()

Query: What is Tokenization?
  BM25   top: Natural-Language-Processing-Python.pdf
  MiniLM top: Natural-Language-Processing-Python.pdf

Query: What are different Types of Tokenization?
  BM25   top: Natural-Language-Processing-Python.pdf
  MiniLM top: Natural-Language-Processing-Python.pdf



## 8. Reflection

- **Tokenization bug:** a plain `.lower().split()` keeps punctuation, so the query token `'tokenization?'` never matched `tokenization` in the PDFs and BM25 ranked the tokenization document *last* for "What is Tokenization?". Stripping punctuation with a regex tokenizer fixes it.
- **BM25** is lexical: it only scores documents where the query words actually appear, so it depends heavily on clean tokenization.
- **MiniLM** is semantic: it compares meaning via embeddings, so it was robust to the punctuation problem and ranked `Natural-Language-Processing-Python.pdf` first even before the fix.
- **After the fix both agree:** BM25 and MiniLM each rank `Natural-Language-Processing-Python.pdf` first for both queries, which is the expected result.